In [2]:
import pandas as pd
from pathlib import Path

# Membaca data Silver
silver = pd.read_csv("../data/silver/silver.csv")

# Memastikan data terbaca
print("Jumlah baris Silver:", len(silver))
print("\nJumlah baris per seri:")
print(silver.groupby("seri_kode").size())

# Memisahkan data provinsi dan Indonesia
data_provinsi = silver[silver["wilayah_kode"] != 9999].copy()
data_indonesia = silver[silver["wilayah_kode"] == 9999].copy()

print("\nJumlah baris provinsi:", len(data_provinsi))
print("Jumlah baris Indonesia:", len(data_indonesia))

Jumlah baris Silver: 1541

Jumlah baris per seri:
seri_kode
FINPROTECTION_CATA_TOT_10_POP     20
HWF_0001                          13
HWF_0006                          13
UHC_INDEX_REPORTED                24
var1402                          393
var1477                          926
var2279                          152
dtype: int64

Jumlah baris provinsi: 1448
Jumlah baris Indonesia: 93


In [3]:
# Memilih hanya tiga seri BPS
seri_bps = ["var1402", "var1477", "var2279"]

bps = silver[silver["seri_kode"].isin(seri_bps)].copy()

# Memisahkan provinsi dan agregat Indonesia
bps_provinsi = bps[bps["wilayah_kode"] != 9999].copy()
bps_indonesia = bps[bps["wilayah_kode"] == 9999].copy()

print("Jumlah baris BPS:", len(bps))
print("Jumlah baris provinsi:", len(bps_provinsi))
print("Jumlah baris Indonesia:", len(bps_indonesia))

print("\nCakupan wilayah per seri dan tahun:")
print(
    bps.groupby(["seri_kode", "periode"])["wilayah_kode"]
    .nunique()
    .unstack(fill_value=0)
)

Jumlah baris BPS: 1471
Jumlah baris provinsi: 1448
Jumlah baris Indonesia: 23

Cakupan wilayah per seri dan tahun:
periode    2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
seri_kode                                                                  
var1402      35    35    35    35    35    35    35    35    35    39    39
var1477       0    34    34    35     0     0     0     0     0     0     0
var2279       0     0     0     0     0     0     0    35    39    39    39


In [4]:
# Membuat daftar seluruh provinsi dari data BPS
daftar_provinsi = (
    bps_provinsi[["wilayah_kode", "nama_wilayah"]]
    .drop_duplicates()
    .sort_values("wilayah_kode")
    .reset_index(drop=True)
)

print("Jumlah provinsi:", len(daftar_provinsi))
display(daftar_provinsi)

Jumlah provinsi: 38


,wilayah_kode,nama_wilayah
0,1100,ACEH
1,1200,SUMATERA UTARA
2,1300,SUMATERA BARAT
3,1400,RIAU
4,1500,JAMBI
5,1600,SUMATERA SELATAN
6,1700,BENGKULU
7,1800,LAMPUNG
8,1900,KEP. BANGKA BELITUNG
9,2100,KEP. RIAU


In [7]:
# Empat provinsi baru di Papua
provinsi_baru = [9200, 9500, 9600, 9700]

# Menyimpan hasil grid setiap seri
hasil_grid = []

for seri in seri_bps:
    data_seri = bps_provinsi[
        bps_provinsi["seri_kode"] == seri
    ].copy()

    periode_seri = sorted(data_seri["periode"].unique())

    for tahun in periode_seri:
        data_tahun = data_seri[
            data_seri["periode"] == tahun
        ].copy()

        # Mengambil kategori rincian yang tersedia pada tahun tersebut
        kategori = data_tahun["rincian"].drop_duplicates().tolist()

        # Membentuk kombinasi 38 provinsi dan kategori
        grid = daftar_provinsi.copy()
        grid["periode"] = tahun

        grid = grid.merge(
            pd.DataFrame({"rincian": kategori}),
            how="cross"
        )

        grid["seri_kode"] = seri

        # Menggabungkan data asli ke grid
        kolom_data = [
            "seri_kode", "wilayah_kode", "periode",
            "rincian", "nilai", "satuan"
        ]

        grid = grid.merge(
            data_tahun[kolom_data],
            on=["seri_kode", "wilayah_kode", "periode", "rincian"],
            how="left",
            validate="one_to_one"
        )

        # Menentukan status setiap sel
        grid["status"] = "terisi"

        belum_terbentuk = (
            grid["wilayah_kode"].isin(provinsi_baru)
            & (grid["periode"] < 2022)
        )

        grid.loc[belum_terbentuk, "status"] = "struktural"

        grid.loc[
            ~belum_terbentuk & grid["nilai"].isna(),
            "status"
        ] = "missing"

        hasil_grid.append(grid)

# Menggabungkan seluruh grid
grid_panel = pd.concat(hasil_grid, ignore_index=True)

print("Jumlah baris grid:", len(grid_panel))
print("\nJumlah status:")
print(grid_panel["status"].value_counts())

display(grid_panel.head(10))

Jumlah baris grid: 1596

Jumlah status:
status
terisi        1448
struktural     136
missing         12
Name: count, dtype: int64


,wilayah_kode,nama_wilayah,periode,rincian,seri_kode,nilai,satuan,status
0,1100,ACEH,2015,Tidak ada,var1402,3.09,Persen,terisi
1,1200,SUMATERA UTARA,2015,Tidak ada,var1402,3.16,Persen,terisi
2,1300,SUMATERA BARAT,2015,Tidak ada,var1402,3.49,Persen,terisi
3,1400,RIAU,2015,Tidak ada,var1402,5.15,Persen,terisi
4,1500,JAMBI,2015,Tidak ada,var1402,4.57,Persen,terisi
5,1600,SUMATERA SELATAN,2015,Tidak ada,var1402,4.33,Persen,terisi
6,1700,BENGKULU,2015,Tidak ada,var1402,5.52,Persen,terisi
7,1800,LAMPUNG,2015,Tidak ada,var1402,5.49,Persen,terisi
8,1900,KEP. BANGKA BELITUNG,2015,Tidak ada,var1402,3.26,Persen,terisi
9,2100,KEP. RIAU,2015,Tidak ada,var1402,3.12,Persen,terisi


In [8]:
# Membuat daftar seluruh provinsi dari data BPS
daftar_provinsi = (
    bps_provinsi[["wilayah_kode", "nama_wilayah"]]
    .drop_duplicates()
    .sort_values("wilayah_kode")
    .reset_index(drop=True)
)

print("Jumlah provinsi:", len(daftar_provinsi))
display(daftar_provinsi)

Jumlah provinsi: 38


,wilayah_kode,nama_wilayah
0,1100,ACEH
1,1200,SUMATERA UTARA
2,1300,SUMATERA BARAT
3,1400,RIAU
4,1500,JAMBI
5,1600,SUMATERA SELATAN
6,1700,BENGKULU
7,1800,LAMPUNG
8,1900,KEP. BANGKA BELITUNG
9,2100,KEP. RIAU


In [9]:
# Memeriksa jumlah kategori tenaga kesehatan per tahun
var1477 = bps_provinsi[
    bps_provinsi["seri_kode"] == "var1477"
].copy()

print("Daftar seluruh kategori:")
print(sorted(var1477["rincian"].unique()))

print("\nJumlah kategori per tahun:")
print(
    var1477.groupby("periode")["rincian"]
    .nunique()
)

print("\nJumlah provinsi per kategori dan tahun:")
print(
    var1477.groupby(["periode", "rincian"])["wilayah_kode"]
    .nunique()
    .unstack(fill_value=0)
)

Daftar seluruh kategori:
['Ahli Teknologi Laboratorium Medik', 'Bidan', 'Dokter Gigi', 'Dokter Umum', 'Farmasi', 'Gizi', 'Kesehatan Lingkungan', 'Kesehatan Masyarakat', 'Pekarya', 'Perawat']

Jumlah kategori per tahun:
periode
2016    10
2017     9
2018     8
Name: rincian, dtype: int64

Jumlah provinsi per kategori dan tahun:
rincian  Ahli Teknologi Laboratorium Medik  Bidan  Dokter Gigi  Dokter Umum  \
periode                                                                       
2016                                    34     34           34           34   
2017                                    34     34           34           34   
2018                                     0     34           34           34   

rincian  Farmasi  Gizi  Kesehatan Lingkungan  Kesehatan Masyarakat  Pekarya  \
periode                                                                       
2016          34    34                    34                    34       34   
2017          34    34                

In [10]:
hasil_grid = []

for seri in seri_bps:
    data_seri = bps_provinsi[
        bps_provinsi["seri_kode"] == seri
    ].copy()

    periode_seri = sorted(data_seri["periode"].unique())

    # Seluruh kategori yang pernah muncul pada seri tersebut
    semua_kategori = data_seri["rincian"].drop_duplicates().tolist()

    for tahun in periode_seri:
        data_tahun = data_seri[
            data_seri["periode"] == tahun
        ].copy()

        kategori_tersedia = data_tahun["rincian"].drop_duplicates().tolist()

        # Membentuk kombinasi 38 provinsi dan seluruh kategori
        grid = daftar_provinsi.copy()
        grid["periode"] = tahun

        grid = grid.merge(
            pd.DataFrame({"rincian": semua_kategori}),
            how="cross"
        )

        grid["seri_kode"] = seri

        # Menggabungkan data asli
        kolom_data = [
            "seri_kode", "wilayah_kode", "periode",
            "rincian", "nilai", "satuan"
        ]

        grid = grid.merge(
            data_tahun[kolom_data],
            on=["seri_kode", "wilayah_kode", "periode", "rincian"],
            how="left",
            validate="one_to_one"
        )

        # Status awal
        grid["status"] = "terisi"

        # Kategori yang tidak diterbitkan pada tahun tersebut
        kategori_tidak_tersedia = ~grid["rincian"].isin(kategori_tersedia)

        grid.loc[
            kategori_tidak_tersedia,
            "status"
        ] = "kategori_tidak_tersedia"

        # Empat provinsi baru sebelum tahun 2022
        wilayah_struktural = (
            grid["wilayah_kode"].isin(provinsi_baru)
            & (grid["periode"] < 2022)
        )

        grid.loc[
            wilayah_struktural,
            "status"
        ] = "struktural"

        # Data yang seharusnya tersedia tetapi nilainya kosong
        kondisi_missing = (
            grid["nilai"].isna()
            & ~kategori_tidak_tersedia
            & ~wilayah_struktural
        )

        grid.loc[kondisi_missing, "status"] = "missing"

        hasil_grid.append(grid)

grid_panel = pd.concat(hasil_grid, ignore_index=True)

print("Jumlah baris grid:", len(grid_panel))
print("\nDistribusi status:")
print(grid_panel["status"].value_counts())

Jumlah baris grid: 1710

Distribusi status:
status
terisi                     1448
struktural                  148
kategori_tidak_tersedia     102
missing                      12
Name: count, dtype: int64


In [11]:
ringkasan_grid = (
    grid_panel
    .groupby(["seri_kode", "periode"])
    .apply(
        lambda x: pd.Series({
            "jumlah_sel": len(x),
            "terisi": (x["status"] == "terisi").sum(),
            "missing": (x["status"] == "missing").sum(),
            "struktural": (x["status"] == "struktural").sum(),
            "kategori_tidak_tersedia": (
                x["status"] == "kategori_tidak_tersedia"
            ).sum(),
            "persen_terisi": (
                (x["status"] == "terisi").sum()
                / (~x["status"].isin([
                    "struktural",
                    "kategori_tidak_tersedia"
                ])).sum()
                * 100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

ringkasan_grid["persen_terisi"] = (
    ringkasan_grid["persen_terisi"].round(2)
)

display(ringkasan_grid)

,seri_kode,periode,jumlah_sel,terisi,missing,struktural,kategori_tidak_tersedia,persen_terisi
0,var1402,2015,38.0,34.0,0.0,4.0,0.0,100.00
1,var1402,2016,38.0,34.0,0.0,4.0,0.0,100.00
2,var1402,2017,38.0,34.0,0.0,4.0,0.0,100.00
3,var1402,2018,38.0,34.0,0.0,4.0,0.0,100.00
4,var1402,2019,38.0,34.0,0.0,4.0,0.0,100.00
5,var1402,2020,38.0,34.0,0.0,4.0,0.0,100.00
6,var1402,2021,38.0,34.0,0.0,4.0,0.0,100.00
7,var1402,2022,38.0,34.0,4.0,0.0,0.0,89.47
8,var1402,2023,38.0,34.0,4.0,0.0,0.0,89.47
9,var1402,2024,38.0,38.0,0.0,0.0,0.0,100.00


In [12]:
# Menampilkan seluruh sel missing
daftar_missing = grid_panel[
    grid_panel["status"] == "missing"
][
    ["seri_kode", "periode", "wilayah_kode", "nama_wilayah", "rincian"]
].sort_values(["seri_kode", "periode", "wilayah_kode"])

display(daftar_missing)

print("Jumlah missing:", len(daftar_missing))

,seri_kode,periode,wilayah_kode,nama_wilayah,rincian
299,var1402,2022,9200,PAPUA BARAT DAYA,Tidak ada
301,var1402,2022,9500,PAPUA SELATAN,Tidak ada
302,var1402,2022,9600,PAPUA TENGAH,Tidak ada
303,var1402,2022,9700,PAPUA PEGUNUNGAN,Tidak ada
337,var1402,2023,9200,PAPUA BARAT DAYA,Tidak ada
339,var1402,2023,9500,PAPUA SELATAN,Tidak ada
340,var1402,2023,9600,PAPUA TENGAH,Tidak ada
341,var1402,2023,9700,PAPUA PEGUNUNGAN,Tidak ada
1591,var2279,2022,9200,PAPUA BARAT DAYA,Tidak ada
1593,var2279,2022,9500,PAPUA SELATAN,Tidak ada


Jumlah missing: 12


In [16]:
# Membuat folder output
from pathlib import Path

ROOT = Path.cwd().parent
print("Folder utama:", ROOT)

folder_output = ROOT / "reports" / "individu"
folder_output.mkdir(parents=True, exist_ok=True)

# Menyimpan grid dan ringkasan
grid_panel.to_csv(
    folder_output / "grid_panel_124450092.csv",
    index=False
)

ringkasan_grid.to_csv(
    folder_output / "kelengkapan_panel_124450092.csv",
    index=False
)

daftar_missing.to_csv(
    folder_output / "missing_panel_124450092.csv",
    index=False
)

print("Tiga file berhasil disimpan.")

Folder utama: c:\Users\User\OneDrive\Documents\GitHub\capstone-dw-kelompok-05
Tiga file berhasil disimpan.


In [17]:
log = pd.read_csv(ROOT / "log-keputusan.csv")

print("Kolom:", log.columns.tolist())
display(log.tail(5))

Kolom: ['id', 'tanggal', 'nim', 'tahap', 'seri_kode', 'keputusan', 'alternatif_ditolak', 'alasan', 'dampak_baris']


,id,tanggal,nim,tahap,seri_kode,keputusan,alternatif_ditolak,alasan,dampak_baris
0,K-001,2026-11-08,124450052,silver,semua_BPS,Kode wilayah provinsi mengikuti kode BPS empat...,Mencampurkan agregat nasional dengan data prov...,Kode wilayah menjaga konsistensi antarseri dan...,0
1,K-002,2026-11-18,124450019,integrasi,var1477,Satuan var1477 ditetapkan sebagai kepadatan pe...,Mengasumsikan nilai var1477 sebagai jumlah ora...,Metadata var menyatakan 'Kepadatan Dan Distrib...,0
2,K-003,2026-11-18,124450019,integrasi,"var1477,HWF_0001,HWF_0006",Untuk perbandingan HWF_0001 dipakai rincian Do...,Menjumlahkan Dokter Umum+Dokter Gigi ke HWF_00...,HWF_0001 = medical doctors (ISCO-08 221) tidak...,0
3,K-004,2026-11-18,124450019,integrasi,"var1402,var2279,UHC_INDEX_REPORTED,FINPROTECTI...",Pasangan var1402-UHC_INDEX_REPORTED dan var227...,Menyetarakan/menyatukan indikator karena sama-...,"Konstruk, skala, arah, dan metode indikator be...",0


In [32]:

# Membaca log terbaru dari file
log = pd.read_csv(ROOT / "log-keputusan.csv")

# Data keputusan baru
keputusan_baru = [
    {
        "id": "K-005",
        "tanggal": "2026-11-19",
        "nim": "124450092",
        "tahap": "grid",
        "seri_kode": "var1402,var2279",
        "keputusan": "Sel kosong pada empat provinsi baru Papua untuk var1402 tahun 2022–2023 dan var2279 tahun 2022 dipertahankan sebagai missing.",
        "alternatif_ditolak": "Mengisi dengan nol, membagi nilai provinsi Papua sebelumnya, atau melakukan imputasi.",
        "alasan": "Data sumber tidak menyediakan nilai terpisah untuk empat provinsi tersebut pada periode terkait sehingga nilai tidak boleh dibuat atau diperkirakan.",
        "dampak_baris": 12
    },
    {
        "id": "K-006",
        "tanggal": "2026-11-19",
        "nim": "124450092",
        "tahap": "grid",
        "seri_kode": "var2279",
        "keputusan": "Grid var2279 dimulai pada tahun 2022 sesuai awal ketersediaan seri.",
        "alternatif_ditolak": "Membentuk grid var2279 untuk tahun sebelum 2022 dan menganggapnya sebagai missing.",
        "alasan": "Seri belum diterbitkan pada tahun-tahun awal sehingga periode sebelum 2022 bukan missing.",
        "dampak_baris": 0
    },
    {
        "id": "K-007",
        "tanggal": "2026-11-19",
        "nim": "124450092",
        "tahap": "grid",
        "seri_kode": "var1477",
        "keputusan": "Kategori var1477 yang tidak muncul pada tahun tertentu dicatat sebagai kategori tidak teramati, bukan diisi nol atau dianggap missing.",
        "alternatif_ditolak": "Mengisi kategori yang tidak muncul dengan nol atau melakukan imputasi.",
        "alasan": "Data sumber tidak memuat observasi kategori tersebut pada tahun terkait dan belum ada bukti bahwa nilai sebenarnya nol.",
        "dampak_baris": 102
    },
    {
        "id": "K-008",
        "tanggal": "2026-11-19",
        "nim": "124450092",
        "tahap": "grid",
        "seri_kode": "var1477",
        "keputusan": "var1477 dipetakan sebagai seri yang relevan dengan SDG 3.c.1 tentang kepadatan dan distribusi tenaga kesehatan.",
        "alternatif_ditolak": "Menyatakan var1477 identik sepenuhnya dengan seluruh komponen SDG 3.c.1.",
        "alasan": "Konsep kepadatan dan distribusi tenaga kesehatan sesuai dengan tema SDG 3.c.1, tetapi cakupan kategori dan definisi sumber perlu diperhatikan. Rujukan: https://unstats.un.org/sdgs/metadata/files/metadata-03-0c-01.pdf",
        "dampak_baris": 0
    }
]

# Hanya tambahkan ID yang belum ada
log_baru = pd.DataFrame(keputusan_baru)
log_baru = log_baru[~log_baru["id"].isin(log["id"])]

log = pd.concat([log, log_baru], ignore_index=True)

# Simpan
log.to_csv(ROOT / "log-keputusan.csv", index=False)

print("Jumlah keputusan:", len(log))
display(log.tail(8))


Jumlah keputusan: 8


,id,tanggal,nim,tahap,seri_kode,keputusan,alternatif_ditolak,alasan,dampak_baris
0,K-001,2026-11-08,124450052,silver,semua_BPS,Kode wilayah provinsi mengikuti kode BPS empat...,Mencampurkan agregat nasional dengan data prov...,Kode wilayah menjaga konsistensi antarseri dan...,0
1,K-002,2026-11-18,124450019,integrasi,var1477,Satuan var1477 ditetapkan sebagai kepadatan pe...,Mengasumsikan nilai var1477 sebagai jumlah ora...,Metadata var menyatakan 'Kepadatan Dan Distrib...,0
2,K-003,2026-11-18,124450019,integrasi,"var1477,HWF_0001,HWF_0006",Untuk perbandingan HWF_0001 dipakai rincian Do...,Menjumlahkan Dokter Umum+Dokter Gigi ke HWF_00...,HWF_0001 = medical doctors (ISCO-08 221) tidak...,0
3,K-004,2026-11-18,124450019,integrasi,"var1402,var2279,UHC_INDEX_REPORTED,FINPROTECTI...",Pasangan var1402-UHC_INDEX_REPORTED dan var227...,Menyetarakan/menyatukan indikator karena sama-...,"Konstruk, skala, arah, dan metode indikator be...",0
4,K-005,2026-11-19,124450092,grid,"var1402,var2279",Sel kosong pada empat provinsi baru Papua untu...,"Mengisi dengan nol, membagi nilai provinsi Pap...",Data sumber tidak menyediakan nilai terpisah u...,12
5,K-006,2026-11-19,124450092,grid,var2279,Grid var2279 dimulai pada tahun 2022 sesuai aw...,Membentuk grid var2279 untuk tahun sebelum 202...,Seri belum diterbitkan pada tahun-tahun awal s...,0
6,K-007,2026-11-19,124450092,grid,var1477,Kategori var1477 yang tidak muncul pada tahun ...,Mengisi kategori yang tidak muncul dengan nol ...,Data sumber tidak memuat observasi kategori te...,102
7,K-008,2026-11-19,124450092,grid,var1477,var1477 dipetakan sebagai seri yang relevan de...,Menyatakan var1477 identik sepenuhnya dengan s...,Konsep kepadatan dan distribusi tenaga kesehat...,0


In [33]:
# Membuat folder output
folder_output = ROOT / "reports" / "individu"
folder_output.mkdir(parents=True, exist_ok=True)

# Menyimpan hasil grid dan ringkasan
grid_panel.to_csv(folder_output / "grid_panel_124450092.csv", index=False)
ringkasan_grid.to_csv(folder_output / "kelengkapan_panel_124450092.csv", index=False)
daftar_missing.to_csv(folder_output / "missing_panel_124450092.csv", index=False)

print("Semua file berhasil disimpan.")

Semua file berhasil disimpan.


In [34]:
display(ringkasan_grid)

,seri_kode,periode,jumlah_sel,terisi,missing,struktural,kategori_tidak_tersedia,persen_terisi
0,var1402,2015,38.0,34.0,0.0,4.0,0.0,100.00
1,var1402,2016,38.0,34.0,0.0,4.0,0.0,100.00
2,var1402,2017,38.0,34.0,0.0,4.0,0.0,100.00
3,var1402,2018,38.0,34.0,0.0,4.0,0.0,100.00
4,var1402,2019,38.0,34.0,0.0,4.0,0.0,100.00
5,var1402,2020,38.0,34.0,0.0,4.0,0.0,100.00
6,var1402,2021,38.0,34.0,0.0,4.0,0.0,100.00
7,var1402,2022,38.0,34.0,4.0,0.0,0.0,89.47
8,var1402,2023,38.0,34.0,4.0,0.0,0.0,89.47
9,var1402,2024,38.0,38.0,0.0,0.0,0.0,100.00


In [35]:
# Menghitung persentase terisi dengan mengecualikan sel struktural
penyebut = ringkasan_grid["jumlah_sel"] - ringkasan_grid["struktural"]

ringkasan_grid["persen_terisi"] = (
    ringkasan_grid["terisi"] / penyebut * 100
).round(2)

# Simpan ulang ringkasan
ringkasan_grid.to_csv(
    folder_output / "kelengkapan_panel_124450092.csv",
    index=False
)

display(ringkasan_grid)

,seri_kode,periode,jumlah_sel,terisi,missing,struktural,kategori_tidak_tersedia,persen_terisi
0,var1402,2015,38.0,34.0,0.0,4.0,0.0,100.00
1,var1402,2016,38.0,34.0,0.0,4.0,0.0,100.00
2,var1402,2017,38.0,34.0,0.0,4.0,0.0,100.00
3,var1402,2018,38.0,34.0,0.0,4.0,0.0,100.00
4,var1402,2019,38.0,34.0,0.0,4.0,0.0,100.00
5,var1402,2020,38.0,34.0,0.0,4.0,0.0,100.00
6,var1402,2021,38.0,34.0,0.0,4.0,0.0,100.00
7,var1402,2022,38.0,34.0,4.0,0.0,0.0,89.47
8,var1402,2023,38.0,34.0,4.0,0.0,0.0,89.47
9,var1402,2024,38.0,38.0,0.0,0.0,0.0,100.00
